# 🗃️ Day 02a: LLD — LRU Cache Design

---

## 🎯 What You'll Master Today
- LRU Cache with OrderedDict
- LRU Cache from scratch (HashMap + Doubly Linked List)
- Thread-safe version
- Design discussion for interviews

**This is LeetCode #146 PLUS a system design discussion.** Two skills in one.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LRU CACHE — DATA STRUCTURE                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Requirement: get() and put() in O(1) time                      ║
║                                                                   ║
║  HashMap alone → O(1) get/put BUT can't track order             ║
║  LinkedList alone → O(1) remove/add BUT O(N) lookup             ║
║  HashMap + Doubly Linked List → O(1) EVERYTHING!                ║
║                                                                   ║
║  HashMap: key → node_ref (O(1) lookup)                          ║
║  DLL: maintains access order (most recent at tail)              ║
║                                                                   ║
║  HEAD ◄──► [A] ◄──► [B] ◄──► [C] ◄──► TAIL                   ║
║  (LRU)               ↑                  (MRU)                    ║
║                      │                                           ║
║            HashMap: {"A": →, "B": →, "C": →}                   ║
║                                                                   ║
║  get(B): move B to tail → O(1)                                  ║
║  put(D): if full, remove head (LRU) → add D at tail → O(1)     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. LRU Cache — OrderedDict (Quick Interview Version)
# ============================================================

from collections import OrderedDict

class LRUCache:
    """O(1) get and put using Python's OrderedDict."""
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)  # Mark as recently used
        return self.cache[key]
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # Remove LRU (first item)

# Test
lru = LRUCache(3)
lru.put(1, "A")
lru.put(2, "B")
lru.put(3, "C")
print(f"get(1): {lru.get(1)}")  # A — moves 1 to end
lru.put(4, "D")                 # Evicts key 2 (LRU)
print(f"get(2): {lru.get(2)}")  # -1 (evicted)
print(f"get(3): {lru.get(3)}")  # C
print(f"Cache: {list(lru.cache.items())}")

In [ ]:
# ============================================================
# 2. LRU Cache — From Scratch (HashMap + DLL)
# ============================================================
# This is what interviewers want to see!

class DLLNode:
    def __init__(self, key=0, val=0):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCacheFromScratch:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key → DLLNode
        
        # Dummy head and tail (sentinel nodes)
        self.head = DLLNode()  # LRU end
        self.tail = DLLNode()  # MRU end
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node: DLLNode):
        """Remove node from its current position in DLL."""
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_tail(self, node: DLLNode):
        """Add node right before tail (MRU position)."""
        node.prev = self.tail.prev
        node.next = self.tail
        self.tail.prev.next = node
        self.tail.prev = node
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self._remove(node)
        self._add_to_tail(node)  # Move to MRU
        return node.val
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self._remove(self.cache[key])
        
        node = DLLNode(key, value)
        self.cache[key] = node
        self._add_to_tail(node)
        
        if len(self.cache) > self.capacity:
            # Remove LRU (node after head)
            lru = self.head.next
            self._remove(lru)
            del self.cache[lru.key]

# Test (same as above)
lru2 = LRUCacheFromScratch(3)
lru2.put(1, "A")
lru2.put(2, "B")
lru2.put(3, "C")
print(f"get(1): {lru2.get(1)}")
lru2.put(4, "D")
print(f"get(2): {lru2.get(2)}")  # -1
print(f"get(3): {lru2.get(3)}")  # C
print(f"\n✅ From-scratch version works identically!")

In [ ]:
# ============================================================
# 3. Thread-Safe LRU Cache
# ============================================================

import threading

class ThreadSafeLRU:
    def __init__(self, capacity):
        self.cache = LRUCacheFromScratch(capacity)
        self.lock = threading.RLock()  # Reentrant lock
    
    def get(self, key):
        with self.lock:
            return self.cache.get(key)
    
    def put(self, key, value):
        with self.lock:
            self.cache.put(key, value)

print("""
THREAD-SAFE CONSIDERATIONS:
═══════════════════════════

Option 1: Global lock (simple, shown above)
  + Easy to implement
  - Bottleneck under high concurrency

Option 2: Read-write lock
  + Multiple concurrent reads
  - BUT get() also writes (moves to tail)! So read lock won't work.

Option 3: Sharded cache (best for distributed)
  + Split into N segments by hash(key) % N
  + Each segment has its own lock
  + N-way parallelism
  This is how ConcurrentHashMap works in Java.

Option 4: Lock-free (CAS-based)
  + Maximum throughput
  - Very complex to implement correctly
  - Used in specialized systems only
""")

In [ ]:
# ============================================================
# 4. Distributed LRU Cache (System Design Discussion)
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  DISTRIBUTED LRU CACHE — SYSTEM DESIGN ANGLE                 ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Single machine: HashMap + DLL (what we built above)         ║
║  Distributed: Multiple cache nodes (Redis cluster)           ║
║                                                                ║
║  Architecture:                                                ║
║  Client → [Consistent Hashing] → Cache Node 1, 2, 3...      ║
║                                                                ║
║  key = hash("user:123") % N → goes to node K                ║
║  Each node runs its own LRU internally                       ║
║                                                                ║
║  Eviction across cluster:                                    ║
║  • Each node evicts independently (per-node LRU)             ║
║  • NOT global LRU (too expensive to coordinate)              ║
║                                                                ║
║  Cold start / Warm-up:                                       ║
║  • New node joins → cache is cold → miss storm               ║
║  • Solution: pre-warm from DB or backup                      ║
║                                                                ║
║  Failure handling:                                            ║
║  • Node dies → consistent hash redirects to next node        ║
║  • Replication: each key stored on N nodes (Redis Cluster)   ║
║                                                                ║
║  Real Redis config:                                           ║
║    maxmemory 4gb                                              ║
║    maxmemory-policy allkeys-lru                               ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Implement LRU Cache (LC #146) | HashMap + DLL. get: move to tail. put: add tail, evict head if full |
| 2 | Why DLL + HashMap? | HashMap = O(1) lookup. DLL = O(1) move/remove. Together = O(1) everything |
| 3 | Why sentinel nodes? | Avoid null checks on head/tail. Simplifies remove/add logic |
| 4 | Thread-safe LRU? | Simple: global lock. Better: sharded cache (N segments, N locks) |
| 5 | LRU vs LFU? | LRU = recency. LFU = frequency. LFU better for stable hot sets |
| 6 | Distributed LRU? | Consistent hashing + per-node LRU. Redis cluster with allkeys-lru |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • LRU = HashMap + Doubly Linked List = O(1) all ops    │
## │  • Sentinel nodes (dummy head/tail) simplify code        │
## │  • OrderedDict for quick version, DLL for full version   │
## │  • Thread-safe: global lock or sharded segments          │
## │  • Distributed: consistent hashing + per-node LRU        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Rate Limiter Design**